In [ ]:
!pip install transformers

!pip uninstall keras tf-keras -y
!pip install tensorflow==2.15.0

# 4. 기타 필요한 패키지 (KoBERT용일 경우)
!pip install gluonnlp pandas tqdm
!pip install mxnet
!pip install sentencepiece

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import re
import os
import urllib.request
from tqdm import tqdm
from transformers import BertTokenizer, TFBertForSequenceClassification

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/전체 데이터_label.csv")
data.head()

In [ ]:
print('총 샘플의 수 :',len(data))

In [ ]:
data

In [ ]:
data[:5]

In [ ]:
del data['keyword']
del data['date']
del data['title']
del data['url']

In [ ]:
data[:5]

In [ ]:
data.info()

In [ ]:
print('결측값 여부 :',data.isnull().values.any())

In [ ]:
print('summary 열의 유니크한 값 :',data['summary'].nunique())

In [ ]:
duplicate = data[data.duplicated()]

In [ ]:
duplicate

In [ ]:
# 중복 제거
data.drop_duplicates(subset=['summary'], inplace=True)
print('총 샘플의 수 :',len(data))

In [ ]:
data['label'].value_counts().plot(kind='bar')

In [ ]:
print('레이블의 분포')
print(data.groupby('label').size().reset_index(name='count'))

In [ ]:
print(f'부정의 비율 = {round(data["label"].value_counts()[0]/len(data) * 100,3)}%')
print(f'중립의 비율 = {round(data["label"].value_counts()[1]/len(data) * 100,3)}%')
print(f'긍정의 비율 = {round(data["label"].value_counts()[2]/len(data) * 100,3)}%')

In [ ]:
data

In [ ]:
X_data = data['summary']
y_data = data['label']
print('본문의 개수: {}'.format(len(X_data)))
print('레이블의 개수: {}'.format(len(y_data)))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=0, stratify=y_data)

In [ ]:
print('훈련 샘플의 개수 :', len(X_train))
print('테스트 샘플의 개수 :', len(X_test))

In [ ]:
print('--------훈련 데이터의 비율-----------')
print(f'부정 = {round(y_train.value_counts()[0]/len(y_train) * 100,3)}%')
print(f'중립 = {round(y_train.value_counts()[1]/len(y_train) * 100,3)}%')
print(f'긍정 = {round(y_train.value_counts()[2]/len(y_train) * 100,3)}%')

In [ ]:
print('--------테스트 데이터의 비율-----------')
print(f'부정 = {round(y_test.value_counts()[0]/len(y_test) * 100,3)}%')
print(f'중립 = {round(y_test.value_counts()[1]/len(y_test) * 100,3)}%')
print(f'긍정 = {round(y_test.value_counts()[2]/len(y_test) * 100,3)}%')

In [ ]:
max_seq_len = 128

In [ ]:
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

In [ ]:
def convert_examples_to_features(examples, labels, max_seq_len, tokenizer):

    input_ids, attention_masks, token_type_ids, data_labels = [], [], [], []

    for example, label in tqdm(zip(examples, labels), total=len(examples)):
        # input_id는 워드 임베딩을 위한 문장의 정수 인코딩
        input_id = tokenizer.encode(example, max_length=max_seq_len, padding='max_length')

        # attention_mask는 실제 단어가 위치하면 1, 패딩의 위치에는 0인 시퀀스.
        padding_count = input_id.count(tokenizer.pad_token_id)
        attention_mask = [1] * (max_seq_len - padding_count) + [0] * padding_count

        # token_type_id은 세그먼트 인코딩
        token_type_id = [0] * max_seq_len

        assert len(input_id) == max_seq_len, "Error with input length {} vs {}".format(len(input_id), max_seq_len)
        assert len(attention_mask) == max_seq_len, "Error with attention mask length {} vs {}".format(len(attention_mask), max_seq_len)
        assert len(token_type_id) == max_seq_len, "Error with token type length {} vs {}".format(len(token_type_id), max_seq_len)

        input_ids.append(input_id)
        attention_masks.append(attention_mask)
        token_type_ids.append(token_type_id)
        data_labels.append(label)

    input_ids = np.array(input_ids, dtype=int)
    attention_masks = np.array(attention_masks, dtype=int)
    token_type_ids = np.array(token_type_ids, dtype=int)

    data_labels = np.asarray(data_labels, dtype=np.int32)

    return (input_ids, attention_masks, token_type_ids), data_labels

In [ ]:
train_X, train_y = convert_examples_to_features(X_train, y_train, max_seq_len=max_seq_len, tokenizer=tokenizer)

In [ ]:
test_X, test_y = convert_examples_to_features(X_test, y_test, max_seq_len=max_seq_len, tokenizer=tokenizer)

In [ ]:
input_id = train_X[0][0]
attention_mask = train_X[1][0]
token_type_id = train_X[2][0]
label = train_y[0]

print('단어에 대한 정수 인코딩 :',input_id)
print('어텐션 마스크 :',attention_mask)
print('세그먼트 인코딩 :',token_type_id)
print('각 인코딩의 길이 :', len(input_id))
print('정수 인코딩 복원 :',tokenizer.decode(input_id))
print('레이블 :',label)

In [ ]:
strategy = tf.distribute.get_strategy()

In [ ]:
import tensorflow as tf
from transformers import TFBertForSequenceClassification
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model = TFBertForSequenceClassification.from_pretrained("klue/bert-base", num_labels=3)

optimizer = Adam(learning_rate=5e-5)
loss_fn = SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])

In [ ]:
print(type(optimizer))

In [ ]:
print(tf.keras.optimizers.Adam)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    min_delta=0.001,
    patience=2)

model.fit(
    train_X, train_y, epochs=3, batch_size=32, validation_split=0.2,
    callbacks = [early_stopping]
)

In [ ]:
model.evaluate(test_X, test_y, batch_size=1024)

In [ ]:
from tensorflow import keras
from transformers import TFBertForSequenceClassification

model = keras.models.load_model(
    '/content/drive/MyDrive/backup_model',
    custom_objects={"TFBertForSequenceClassification": TFBertForSequenceClassification}
)

In [ ]:
# 모델을 Drive에 저장
# model.save('/content/drive/MyDrive/my_model.keras')

In [ ]:
# model.save('/content/drive/MyDrive/backup_model', save_format='tf')

In [ ]:
from sklearn.metrics import classification_report, f1_score

# test_X를 이용한 예측
predictions = model.predict(test_X).logits
predicted_labels = np.argmax(predictions, axis=1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y, predicted_labels, digits=4))